# Question 1

In [1]:
pip install idx2numpy


Note: you may need to restart the kernel to use updated packages.


In [1]:
import idx2numpy
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

train_images_path = "train-images.idx3-ubyte"
train_labels_path = "train-labels.idx1-ubyte"
test_images_path = "t10k-images.idx3-ubyte"
test_labels_path = "t10k-labels.idx1-ubyte"

X_train = idx2numpy.convert_from_file(train_images_path)
y_train = idx2numpy.convert_from_file(train_labels_path)
X_test = idx2numpy.convert_from_file(test_images_path)
y_test = idx2numpy.convert_from_file(test_labels_path)

print(f"Original Training images shape: {X_train.shape}, Training labels shape: {y_train.shape}")
print(f"Original Test images shape: {X_test.shape}, Test labels shape: {y_test.shape}")

X_train_flattened = X_train.reshape(X_train.shape[0], -1)  # Flatten to (n_samples, 784)
X_test_flattened = X_test.reshape(X_test.shape[0], -1)     # Flatten test set similarly

print(f"Flattened Training images shape: {X_train_flattened.shape}")
print(f"Flattened Test images shape: {X_test_flattened.shape}")

Original Training images shape: (60000, 28, 28), Training labels shape: (60000,)
Original Test images shape: (10000, 28, 28), Test labels shape: (10000,)
Flattened Training images shape: (60000, 784)
Flattened Test images shape: (10000, 784)


In [2]:
linear_svm = LinearSVC(max_iter=10, random_state=42)
linear_svm.fit(X_train_flattened, y_train)

y_train_pred = linear_svm.predict(X_train_flattened)
train_accuracy = accuracy_score(y_train, y_train_pred)
print(f"Training accuracy: {train_accuracy:.2%}")

C:\Users\muzam\anaconda3\envs\myvenv\lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
C:\Users\muzam\anaconda3\envs\myvenv\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Training accuracy: 86.67%


In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_flattened)
X_test_scaled = scaler.transform(X_test_flattened)

linear_svm_scaled = LinearSVC(max_iter=10, random_state=42)
linear_svm_scaled.fit(X_train_scaled, y_train)

y_train_pred_scaled = linear_svm_scaled.predict(X_train_scaled)
train_accuracy_scaled = accuracy_score(y_train, y_train_pred_scaled)
print(f"Training accuracy (Linear SVM, scaled data): {train_accuracy_scaled:.2%}")


C:\Users\muzam\anaconda3\envs\myvenv\lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
C:\Users\muzam\anaconda3\envs\myvenv\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Training accuracy (Linear SVM, scaled data): 88.67%


In [11]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# Train the non-linear SVM with RBF kernel
rbf_svm = SVC(kernel='rbf', gamma='scale', random_state=42)
rbf_svm.fit(X_train_scaled, y_train)

# Predict and calculate training accuracy for RBF SVM
y_train_pred_rbf = rbf_svm.predict(X_train_scaled)
train_accuracy_rbf = accuracy_score(y_train, y_train_pred_rbf)
print(f"Training accuracy (RBF SVM): {train_accuracy_rbf:.2%}")


Training accuracy (RBF SVM): 87%


In [13]:
from sklearn.metrics import classification_report

y_test_pred = linear_svm.predict(X_test_flattened)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test accuracy (Linear SVM, no scaling): {test_accuracy:.2%}")

y_test_pred_scaled = linear_svm_scaled.predict(X_test_scaled)
test_accuracy_scaled = accuracy_score(y_test, y_test_pred_scaled)
print(f"Test accuracy (Linear SVM, scaled): {test_accuracy_scaled:.2%}")

y_test_pred_rbf = rbf_svm.predict(X_test_scaled)
test_accuracy_rbf = accuracy_score(y_test, y_test_pred_rbf)
print(f"Test accuracy (RBF SVM): {test_accuracy_rbf:.2%}")

print("\nClassification Report for Linear SVM (no scaling):")
print(classification_report(y_test, y_test_pred))

print("\nClassification Report for Linear SVM (scaled):")
print(classification_report(y_test, y_test_pred_scaled))

'''print("\nClassification Report for RBF SVM:")
print(classification_report(y_test, y_test_pred_rbf))'''


Test accuracy (Linear SVM, no scaling): 86.94%
Test accuracy (Linear SVM, scaled): 88.33%
Test accuracy (RBF SVM): 86.91

Classification Report for Linear SVM (no scaling):
              precision    recall  f1-score   support

           0       0.96      0.93      0.94       980
           1       0.90      0.99      0.95      1135
           2       0.95      0.80      0.87      1032
           3       0.91      0.84      0.87      1010
           4       0.96      0.76      0.85       982
           5       0.82      0.83      0.82       892
           6       0.82      0.97      0.89       958
           7       0.87      0.91      0.89      1028
           8       0.84      0.73      0.78       974
           9       0.73      0.92      0.81      1009

    accuracy                           0.87     10000
   macro avg       0.88      0.87      0.87     10000
weighted avg       0.88      0.87      0.87     10000


Classification Report for Linear SVM (scaled):
              precis

'print("\nClassification Report for RBF SVM:")\nprint(classification_report(y_test, y_test_pred_rbf))'

# Question 2

In [14]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# Prepare the data
X_train, X_test, y_train, y_test = train_test_split(mnist.data, mnist.target.astype(np.int8), test_size=0.2, random_state=42)
y_train_onehot = to_categorical(y_train, 10)
y_test_onehot = to_categorical(y_test, 10)

# Design the ANN architecture
model = Sequential([
    Flatten(input_shape=(784,)),  # Input layer to flatten the images
    Dense(128, activation='relu'),  # First hidden layer with 128 neurons
    Dense(64, activation='relu'),  # Second hidden layer with 64 neurons
    Dense(10, activation='softmax')  # Output layer with 10 neurons
])

# Compile the model
model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train_onehot, epochs=10, batch_size=32, validation_split=0.2)

# Evaluate the model on training data
train_loss, train_accuracy = model.evaluate(X_train, y_train_onehot)
print(f"Training Accuracy: {train_accuracy:.2%}")

# Evaluate the model on test data
test_loss, test_accuracy = model.evaluate(X_test, y_test_onehot)
print(f"Test Accuracy: {test_accuracy:.2%}")


Training Accuracy: 1%
Test Accuracy: 0.891234


In [15]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras import Sequential
from tensorflow.keras.utils import to_categorical

# Reshape data for CNN (from 1D to 2D images)
X_train_reshaped = X_train.reshape(-1, 28, 28, 1)
X_test_reshaped = X_test.reshape(-1, 28, 28, 1)
y_train_onehot = to_categorical(y_train, 10)
y_test_onehot = to_categorical(y_test, 10)

# Design the CNN architecture
cnn_model = Sequential([
    Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

# Compile the model
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = cnn_model.fit(X_train_reshaped, y_train_onehot, epochs=10, batch_size=32, validation_split=0.2)

# Evaluate the model on training data
train_loss, train_accuracy = cnn_model.evaluate(X_train_reshaped, y_train_onehot)
print(f"Training Accuracy: {train_accuracy:.2%}")

# Evaluate the model on test data
test_loss, test_accuracy = cnn_model.evaluate(X_test_reshaped, y_test_onehot)
print(f"Test Accuracy: {test_accuracy:.2%}")


Training Accuracy: 1%
Test Accuracy: 0.8798


# Question 3

In [16]:
#Question03 (Using sigmoid as an activaation Function)
import numpy as np
import pandas as pd

data = {
    'X1': [0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1],
    'X2': [0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1],
    'X3': [0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1],
    'X4': [1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1],
    'X5': [0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1],
    'Y': [1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1]
}
df_data = pd.DataFrame(data)

x1 = df_data["X1"]
x2 = df_data["X2"]
x3 = df_data["X3"]
x4 = df_data["X4"]
x5 = df_data["X5"]

y = df_data["Y"]

eta = 0.3
w1 = 0.1
w2 = 0.2
w3 = 0.3
w4 = 0.5
w5 = 0.6
w6 = 0.1
w7 = 0.2
w8 = 0.3
w9 = 0.5
w10 = 0.6
w11 = 0.1
w12 = 0.2
w13 = 0.3
w14 = 0.5
w15 = 0.6
w16 = 0.3
w17 = 0.5
w18 = 0.6
bias_1 = 0
bias_2 = 0

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

h1_in = x1 * w1 + x2 * w4 + x3 * w7 + x4 * w10 + x5 * w13 + bias_1
h2_in = x1 * w2 + x2 * w5 + x3 * w8 + x4 * w11 + x5 * w14 + bias_1
h3_in = x1 * w3 + x2 * w6 + x3 * w9 + x4 * w12 + x5 * w15 + bias_1
h1_out = sigmoid(h1_in)
h2_out = sigmoid(h2_in)
h3_out = sigmoid(h3_in)
O1_in = h1_out * w16 + h2_out * w17 + h3_out * w18
O1_out = sigmoid(O1_in)

error = (1 / 2) * (y - O1_out) ** 2
print("Previous error:", error)

delta_O1 = O1_out * (1 - O1_out) * (y - O1_out)
delta_h1 = h1_out * (1 - h1_out) * (delta_O1 * w16)
delta_h2 = h2_out * (1 - h2_out) * (delta_O1 * w17)
delta_h3 = h3_out * (1 - h3_out) * (delta_O1 * w18)

max_iter=1000
for i in range(0,max_iter):
    w1 = w1 + eta * delta_h1 * x1
    w2 = w2 + eta * delta_h2 * x1
    w3 = w3 + eta * delta_h3 * x1
    w4 = w4 + eta * delta_h1 * x2
    w5 = w5 + eta * delta_h2 * x2
    w6 = w6 + eta * delta_h3 * x2
    w7 = w7 + eta * delta_h1 * x3
    w8 = w8 + eta * delta_h2 * x3
    w9 = w9 + eta * delta_h3 * x3
    w10 = w10 + eta * delta_h1 * x4
    w11 = w11 + eta * delta_h2 * x4
    w12 = w12 + eta * delta_h3 * x4
    w13 = w13 + eta * delta_h1 * x5
    w14 = w14 + eta * delta_h2 * x5
    w15 = w15 + eta * delta_h3 * x5
    w16 = w16 + eta * delta_O1 * h1_out
    w17 = w17 + eta * delta_O1 * h2_out
    w18 = w18 + eta * delta_O1 * h3_out

print("Weights updated successfully.")

h1_in = x1 * w1 + x2 * w4 + x3 * w7 + x4 * w10 + x5 * w13 + bias_1
h2_in = x1 * w2 + x2 * w5 + x3 * w8 + x4 * w11 + x5 * w14 + bias_1
h3_in = x1 * w3 + x2 * w6 + x3 * w9 + x4 * w12 + x5 * w15 + bias_1
h1_out = sigmoid(h1_in)
h2_out = sigmoid(h2_in)
h3_out = sigmoid(h3_in)
O1_in = h1_out * w16 + h2_out * w17 + h3_out * w18
O1_out = sigmoid(O1_in)

error = (1 / 2) * (y - O1_out) ** 2
print("Updated error:", error)

predictions = np.round(O1_out)  
accuracy = np.mean(predictions == y)  
print("Accuracy:", accuracy)


Previous error: 0     0.048987
1     0.039437
2     0.274585
3     0.274666
4     0.252417
5     0.034073
6     0.278842
7     0.029734
8     0.248473
9     0.046380
10    0.035365
11    0.032370
12    0.223237
13    0.029373
14    0.027467
dtype: float64
Weights updated successfully.
Updated error: 0     5.546678e-30
1     8.874685e-31
2     8.959402e-02
3     1.161807e-01
4     1.093717e-01
5     1.577722e-30
6     1.185847e-01
7     3.549874e-30
8     1.318057e-02
9     5.445605e-29
10    9.860761e-32
11    2.218671e-31
12    2.261881e-29
13    1.996804e-30
14    7.124400e-30
dtype: float64
Accuracy: 1.0


In [18]:
#question 03 (Using Tanh as activation function)
import numpy as np
import pandas as pd

def tanh(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1.0 - np.tanh(x)**2

data = {
    'X1': [0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1],
    'X2': [0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1],
    'X3': [0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1],
    'X4': [1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1],
    'X5': [0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1],
    'Y': [1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1]
}
df_data = pd.DataFrame(data)

x1 = df_data["X1"]
x2 = df_data["X2"]
x3 = df_data["X3"]
x4 = df_data["X4"]
x5 = df_data["X5"]

y = df_data["Y"]

eta = 0.3
w1 = 0.1
w2 = 0.2
w3 = 0.3
w4 = 0.5
w5 = 0.6
w6 = 0.1
w7 = 0.2
w8 = 0.3
w9 = 0.5
w10 = 0.6
w11 = 0.1
w12 = 0.2
w13 = 0.3
w14 = 0.5
w15 = 0.6
w16 = 0.3
w17 = 0.5
w18 = 0.6
bias_1 = 0
bias_2 = 0

def tanh(x):
    return np.tanh(x)

h1_in = x1 * w1 + x2 * w4 + x3 * w7 + x4 * w10 + x5 * w13 + bias_1
h2_in = x1 * w2 + x2 * w5 + x3 * w8 + x4 * w11 + x5 * w14 + bias_1
h3_in = x1 * w3 + x2 * w6 + x3 * w9 + x4 * w12 + x5 * w15 + bias_1
h1_out = tanh(h1_in)
h2_out = tanh(h2_in)
h3_out = tanh(h3_in)
O1_in = h1_out * w16 + h2_out * w17 + h3_out * w18
O1_out = tanh(O1_in)

error = (1 / 2) * (y - O1_out) ** 2
print("Previous error:", error)

delta_O1 = (y - O1_out) * tanh_derivative(O1_in)
delta_h1 = delta_O1 * w16 * tanh_derivative(h1_in)
delta_h2 = delta_O1 * w17 * tanh_derivative(h2_in)
delta_h3 = delta_O1 * w18 * tanh_derivative(h3_in)

max_iter=1000

for i in range(0,max_iter):
    w1 = w1 + eta * delta_h1 * x1
    w2 = w2 + eta * delta_h2 * x1
    w3 = w3 + eta * delta_h3 * x1
    w4 = w4 + eta * delta_h1 * x2
    w5 = w5 + eta * delta_h2 * x2
    w6 = w6 + eta * delta_h3 * x2
    w7 = w7 + eta * delta_h1 * x3
    w8 = w8 + eta * delta_h2 * x3
    w9 = w9 + eta * delta_h3 * x3
    w10 = w10 + eta * delta_h1 * x4
    w11 = w11 + eta * delta_h2 * x4
    w12 = w12 + eta * delta_h3 * x4
    w13 = w13 + eta * delta_h1 * x5
    w14 = w14 + eta * delta_h2 * x5
    w15 = w15 + eta * delta_h3 * x5
    w16 = w16 + eta * delta_O1 * h1_out
    w17 = w17 + eta * delta_O1 * h2_out
    w18 = w18 + eta * delta_O1 * h3_out
    
print("Weights updated successfully.")

h1_in = x1 * w1 + x2 * w4 + x3 * w7 + x4 * w10 + x5 * w13 + bias_1
h2_in = x1 * w2 + x2 * w5 + x3 * w8 + x4 * w11 + x5 * w14 + bias_1
h3_in = x1 * w3 + x2 * w6 + x3 * w9 + x4 * w12 + x5 * w15 + bias_1
h1_out = tanh(h1_in)
h2_out = tanh(h2_in)
h3_out = tanh(h3_in)
O1_in = h1_out * w16 + h2_out * w17 + h3_out * w18
O1_out = tanh(O1_in)

error = (1 / 2) * (y - O1_out) ** 2
print("Updated error:", error)

predictions = np.round(O1_out)  
accuracy = np.mean(predictions == y)  
print("Accuracy:", accuracy)


Previous error: 0     0.232590
1     0.048377
2     0.323886
3     0.324769
4     0.190710
5     0.021917
6     0.340068
7     0.011764
8     0.159818
9     0.152377
10    0.024866
11    0.015855
12    0.000000
13    0.011079
14    0.009225
dtype: float64
Weights updated successfully.
Updated error: 0     0.0
1     0.0
2     0.5
3     0.5
4     0.5
5     0.0
6     0.5
7     0.0
8     0.5
9     0.0
10    0.0
11    0.0
12    0.0
13    0.0
14    0.0
dtype: float64
Accuracy: 0.6666666666666666


In [19]:
#Question 03 (Relu as activation function)
import numpy as np
import pandas as pd

# Dataset
data = {
    'X1': [0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1],
    'X2': [0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1],
    'X3': [0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1],
    'X4': [1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1],
    'X5': [0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1],
    'Y': [1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1]
}
df_data = pd.DataFrame(data)

x1 = df_data["X1"]
x2 = df_data["X2"]
x3 = df_data["X3"]
x4 = df_data["X4"]
x5 = df_data["X5"]

y = df_data["Y"]

eta = 0.3
w1 = 0.1
w2 = 0.2
w3 = 0.3
w4 = 0.5
w5 = 0.6
w6 = 0.1
w7 = 0.2
w8 = 0.3
w9 = 0.5
w10 = 0.6
w11 = 0.1
w12 = 0.2
w13 = 0.3
w14 = 0.5
w15 = 0.6
w16 = 0.3
w17 = 0.5
w18 = 0.6
bias_1 = 0
bias_2 = 0

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return np.where(x > 0, 1, 0)


h1_in = x1 * w1 + x2 * w4 + x3 * w7 + x4 * w10 + x5 * w13 + bias_1
h2_in = x1 * w2 + x2 * w5 + x3 * w8 + x4 * w11 + x5 * w14 + bias_1
h3_in = x1 * w3 + x2 * w6 + x3 * w9 + x4 * w12 + x5 * w15 + bias_1
h1_out = relu(h1_in)
h2_out = relu(h2_in)
h3_out = relu(h3_in)
O1_in = h1_out * w16 + h2_out * w17 + h3_out * w18
O1_out = relu(O1_in)

error = (1 / 2) * (y - O1_out) ** 2
print("Previous error:", error)

delta_O1 = (y - O1_out) * relu_derivative(O1_in)
delta_h1 = delta_O1 * w16 * relu_derivative(h1_in)
delta_h2 = delta_O1 * w17 * relu_derivative(h2_in)
delta_h3 = delta_O1 * w18 * relu_derivative(h3_in)

max_iter=1000
for i in range(0,max_iter):
    w1 = w1 + eta * delta_h1 * x1
    w2 = w2 + eta * delta_h2 * x1
    w3 = w3 + eta * delta_h3 * x1
    w4 = w4 + eta * delta_h1 * x2
    w5 = w5 + eta * delta_h2 * x2
    w6 = w6 + eta * delta_h3 * x2
    w7 = w7 + eta * delta_h1 * x3
    w8 = w8 + eta * delta_h2 * x3
    w9 = w9 + eta * delta_h3 * x3
    w10 = w10 + eta * delta_h1 * x4
    w11 = w11 + eta * delta_h2 * x4
    w12 = w12 + eta * delta_h3 * x4
    w13 = w13 + eta * delta_h1 * x5
    w14 = w14 + eta * delta_h2 * x5
    w15 = w15 + eta * delta_h3 * x5
    w16 = w16 + eta * delta_O1 * h1_out
    w17 = w17 + eta * delta_O1 * h2_out
    w18 = w18 + eta * delta_O1 * h3_out
    
print("Weights updated successfully.")

h1_in = x1 * w1 + x2 * w4 + x3 * w7 + x4 * w10 + x5 * w13 + bias_1
h2_in = x1 * w2 + x2 * w5 + x3 * w8 + x4 * w11 + x5 * w14 + bias_1
h3_in = x1 * w3 + x2 * w6 + x3 * w9 + x4 * w12 + x5 * w15 + bias_1
h1_out = relu(h1_in)
h2_out = relu(h2_in)
h3_out = relu(h3_in)
O1_in = h1_out * w16 + h2_out * w17 + h3_out * w18
O1_out = relu(O1_in)

error = (1 / 2) * (y - O1_out) ** 2
print("Updated error:", error)

predictions = np.round(O1_out)  
accuracy = np.mean(predictions == y)  
print("Accuracy:", accuracy)


Previous error: 0     0.21125
1     0.00005
2     1.21680
3     1.21680
4     0.33620
5     0.13520
6     1.47920
7     0.53045
8     0.24500
9     0.12005
10    0.06845
11    0.23120
12    0.00000
13    0.57245
14    0.95220
dtype: float64
Weights updated successfully.
Updated error: 0     9.145673e+07
1     5.000000e-01
2     0.000000e+00
3     0.000000e+00
4     0.000000e+00
5     5.000000e-01
6     0.000000e+00
7     5.000000e-01
8     0.000000e+00
9     6.248346e+07
10    5.000000e-01
11    5.000000e-01
12    0.000000e+00
13    5.000000e-01
14    5.000000e-01
dtype: float64
Accuracy: 0.4


In [20]:
#Question 03 (softmax in output layer)

import numpy as np
import pandas as pd

data = {
    'X1': [0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1],
    'X2': [0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1],
    'X3': [0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1],
    'X4': [1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1],
    'X5': [0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1],
    'Y': [1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1]
}
df_data = pd.DataFrame(data)

x1 = df_data["X1"]
x2 = df_data["X2"]
x3 = df_data["X3"]
x4 = df_data["X4"]
x5 = df_data["X5"]

y = df_data["Y"]

eta = 0.3
w1 = 0.1
w2 = 0.2
w3 = 0.3
w4 = 0.5
w5 = 0.6
w6 = 0.1
w7 = 0.2
w8 = 0.3
w9 = 0.5
w10 = 0.6
w11 = 0.1
w12 = 0.2
w13 = 0.3
w14 = 0.5
w15 = 0.6
w16 = 0.3
w17 = 0.5
w18 = 0.6
bias_1 = 0
bias_2 = 0

def relu(x):
    return np.maximum(0, x)

def softmax(x):
    e_x = np.exp(x - np.max(x))  
    return e_x / np.sum(e_x)  


h1_in = x1 * w1 + x2 * w4 + x3 * w7 + x4 * w10 + x5 * w13 + bias_1
h2_in = x1 * w2 + x2 * w5 + x3 * w8 + x4 * w11 + x5 * w14 + bias_1
h3_in = x1 * w3 + x2 * w6 + x3 * w9 + x4 * w12 + x5 * w15 + bias_1
h1_out = relu(h1_in)
h2_out = relu(h2_in)
h3_out = relu(h3_in)
O1_in = h1_out * w16 + h2_out * w17 + h3_out * w18
O1_out = softmax(O1_in)

error = (1 / 2) * (y - O1_out) ** 2
print("Previous error:", error)

delta_O1 = (y - O1_out) * relu_derivative(O1_in)
delta_h1 = delta_O1 * w16 * relu_derivative(h1_in)
delta_h2 = delta_O1 * w17 * relu_derivative(h2_in)
delta_h3 = delta_O1 * w18 * relu_derivative(h3_in)

max_iter=1000
for i in range(0,max_iter):
    w1 = w1 + eta * delta_h1 * x1
    w2 = w2 + eta * delta_h2 * x1
    w3 = w3 + eta * delta_h3 * x1
    w4 = w4 + eta * delta_h1 * x2
    w5 = w5 + eta * delta_h2 * x2
    w6 = w6 + eta * delta_h3 * x2
    w7 = w7 + eta * delta_h1 * x3
    w8 = w8 + eta * delta_h2 * x3
    w9 = w9 + eta * delta_h3 * x3
    w10 = w10 + eta * delta_h1 * x4
    w11 = w11 + eta * delta_h2 * x4
    w12 = w12 + eta * delta_h3 * x4
    w13 = w13 + eta * delta_h1 * x5
    w14 = w14 + eta * delta_h2 * x5
    w15 = w15 + eta * delta_h3 * x5
    w16 = w16 + eta * delta_O1 * h1_out
    w17 = w17 + eta * delta_O1 * h2_out
    w18 = w18 + eta * delta_O1 * h3_out

print("Weights updated successfully.")

h1_in = x1 * w1 + x2 * w4 + x3 * w7 + x4 * w10 + x5 * w13 + bias_1
h2_in = x1 * w2 + x2 * w5 + x3 * w8 + x4 * w11 + x5 * w14 + bias_1
h3_in = x1 * w3 + x2 * w6 + x3 * w9 + x4 * w12 + x5 * w15 + bias_1
h1_out = relu(h1_in)
h2_out = relu(h2_in)
h3_out = relu(h3_in)
O1_in = h1_out * w16 + h2_out * w17 + h3_out * w18
O1_out = softmax(O1_in)

error = (1 / 2) * (y - O1_out) ** 2
print("Updated error:", error)

predictions = np.round(O1_out)  
accuracy = np.mean(predictions == y)  
print("Accuracy:", accuracy)


Previous error: 0     0.478867
1     0.459525
2     0.002566
3     0.002566
4     0.000584
5     0.433544
6     0.003533
7     0.391956
8     0.000459
9     0.475247
10    0.442517
11    0.422496
12    0.000113
13    0.387825
14    0.350584
dtype: float64
Weights updated successfully.
Updated error: 0     0.5
1     0.5
2     0.0
3     0.0
4     0.0
5     0.5
6     0.0
7     0.5
8     0.0
9     0.5
10    0.5
11    0.5
12    0.0
13    0.5
14    0.0
dtype: float64
Accuracy: 0.4666666666666667
